# SPV-MIA Adapted to Federated LLM Fine-Tuning

This notebook adapts **SPV-MIA** (Self-calibrated Probabilistic Variation MIA; Fu et al.,
*Practical Membership Inference Attacks against Fine-tuned Large Language Models via Self-prompt
Calibration*, NeurIPS 2024, arXiv:2311.06062; code
https://github.com/tsinghua-fib-lab/NeurIPS2024_SPV-MIA) to federated learning (FL) based
fine-tuning of a causal language model.

**Original attack.** Two modules combined into `A(x, theta, theta_dot) = 1[ p_tilde_theta(x) -
p_tilde_theta_dot(x) >= tau ]`:

1. **Self-prompt reference model** — instead of a real reference dataset, prompt the *target* LLM
   with short public-domain chunks so it generates a self-dataset `D_self ~ p_theta`, then
   fine-tune a reference model `theta_dot` on `D_self` (practical difficulty calibration,
   `Delta_m(x) = m_theta(x) - m_theta_dot(x)`).
2. **Probabilistic Variation Assessment** — detect whether `x` is a **local maximum** of the
   model's probability surface (memorization signature) via symmetric paraphrases:
   `p_tilde_theta(x) ~= mean_n( p_theta(x_n^+) + p_theta(x_n^-) )/2 - p_theta(x)`.

**FL adaptation.** Construct paired positive / negative worlds where a target record is either
included in a target client's local fine-tuning data or absent. Federated fine-tuning (FedAvg)
runs first to obtain the target model `theta`. The server-side attacker then **bootstraps the
self-prompt reference `theta_dot`** by prompting the FL-fine-tuned model to generate a self-dataset
and fitting a second model on it, computes the probabilistic variation of the target record under
both `theta` and `theta_dot`, and thresholds `pv_theta - pv_theta_dot`. It reports TPR / TNR / Adv
plus a threshold-free ROC-AUC (the paper's AUC) and persists results in Firebase Firestore.

**Sign convention (documented).** Paper Eq. 10 sets `p_tilde(x) = mean(paraphrase_probs) -
prob_x`, which is negative for a memorized local maximum. To keep the project convention that a
**higher score means a more likely member**, this notebook uses the sign-flipped memorization
signal `pv(x) = prob_x - mean(paraphrase_probs) = -p_tilde(x)`, so `spv_membership_score =
pv_theta(x) - pv_theta_dot(x)` is higher for members. Orientation only; ranking and AUC are
unchanged.

**Relation to the project's main paper.** Of the ten attacks, SPV-MIA is the closest to the main
paper's federated / differential-privacy privacy-assessment goal in the LLM fine-tuning setting: it
is a passive, black-box leakage *measurement* with an explicit DP-SGD-vs-AUC trade-off (AUC ~0.87
at eps=60, ~0.77 at eps=15), making it a natural instrument for judging whether privacy improved or
declined after adding DP, changing the PEFT method, or applying a defense in a federated
fine-tuning environment.

## Configuration

Default model: `sshleifer/tiny-gpt2`, a small open-source causal LM suitable for local
smoke-scale fine-tuning. Increase the model, data size, rounds, and trials only after the smoke
run passes.

SPV-specific knobs: `num_paraphrases` symmetric paraphrases per record, `mask_ratio` (~0.2 of
tokens masked and reconstructed), and `self_prompt_tokens` (length of the public-domain prompt
chunk used to generate the self-dataset for the reference model).

The decision `threshold` is on `spv_membership_score = pv_theta - pv_theta_dot`. Because the
score's absolute scale depends on the model, calibrate it on a held-out split for a real run or
report threshold-free ROC-AUC (the paper's AUC).

## GPU Selection

On a shared multi-GPU box, pin this notebook to a single GPU **before** any CUDA
initialization to avoid out-of-memory crashes from other jobs. This cell must run
first (top to bottom). It picks the GPU as follows:

1. `EXPERIMENT_GPU` from the shell environment, else from `.env` (e.g. `EXPERIMENT_GPU=1`).
2. Otherwise auto-selects the GPU with the most free memory (via `nvidia-smi`).

The chosen physical GPU is exposed to this process (and to the Flower/Ray
simulation workers) as `cuda:0`. Set `EXPERIMENT_GPU=cpu` to force CPU.

In [ ]:
# Pin to one GPU before torch initializes CUDA (avoids OOM on a shared box).
import os
import shutil
import subprocess
from pathlib import Path


def _read_env_file_var(name: str):
    # Minimal .env reader so GPU pinning works before python-dotenv is loaded.
    for base in [Path.cwd(), *Path.cwd().parents]:
        env_path = base / ".env"
        if env_path.exists():
            for line in env_path.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                if key.strip() == name:
                    return value.strip().strip('"').strip("'")
            break
    return None


def _gpu_free_memory():
    if shutil.which("nvidia-smi") is None:
        return []
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=index,memory.free", "--format=csv,noheader,nounits"],
            text=True,
        )
    except Exception:
        return []
    rows = []
    for line in out.strip().splitlines():
        if not line.strip():
            continue
        idx, free = line.split(",")
        rows.append((idx.strip(), int(free)))
    return rows


def select_gpu() -> str | None:
    forced = os.environ.get("EXPERIMENT_GPU") or _read_env_file_var("EXPERIMENT_GPU")
    if forced is not None and forced.strip() != "":
        return forced.strip()
    rows = _gpu_free_memory()
    if not rows:
        return None
    return max(rows, key=lambda r: r[1])[0]


_gpu = select_gpu()
if _gpu is None:
    print("GPU selection: no GPU detected; using default device visibility.")
elif _gpu.lower() == "cpu":
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    print("GPU selection: forced CPU (CUDA_VISIBLE_DEVICES='').")
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = _gpu
    _free = dict(_gpu_free_memory()).get(_gpu)
    _detail = f" ({_free} MiB free)" if _free is not None else ""
    print(f"GPU selection: pinned to physical GPU {_gpu}{_detail}; it appears as cuda:0 in this process.")

In [ ]:
from dataclasses import asdict, dataclass, replace
from hashlib import sha256
from itertools import product
from pathlib import Path
from statistics import mean
from typing import Dict, Iterable, List, Optional, Sequence
import json
import math
import os
import random
import shutil
import time

@dataclass(frozen=True)
class ExperimentConfig:
    attack_name: str = "spv_mia"
    paper_source: str = "Fu et al. 2024 (NeurIPS, arXiv:2311.06062) SPV-MIA: self-prompt calibration + probabilistic variation"
    model_id: str = "sshleifer/tiny-gpt2"
    dataset_name: str = "synthetic_client_text"
    num_clients: int = 4
    clients_per_round: int = 4
    federated_rounds: int = 1
    local_epochs: int = 1
    local_batch_size: int = 2
    client_lr: float = 5e-5
    target_client_id: int = 0
    attack_trials: int = 4
    # Threshold on spv_membership_score = pv_theta - pv_theta_dot (higher => member).
    # A member is a local maximum under theta but not under the self-prompt reference,
    # so its calibrated score is positive; non-members are ~0/negative. Zero separates
    # cleanly at smoke scale; calibrate on a held-out split for a real run.
    threshold: float = 0.0
    max_length: int = 64
    # SPV-MIA specific knobs.
    num_paraphrases: int = 4
    mask_ratio: float = 0.2
    self_prompt_tokens: int = 8
    seed: int = 7
    firestore_collection: str = "ami_federated_llm_results"
    artifact_root: str = "artifacts/spv_mia_adaptation"
    fl_framework: str = "flower"
    sim_num_gpus: float = 0.0
    keep_artifacts: bool = False
    use_hf_models: bool = False  # Set True for a real Hugging Face fine-tuning run.

BASE_CONFIG = ExperimentConfig()

SWEEP = {
    "model_id": ["sshleifer/tiny-gpt2"],
    "federated_rounds": [1],
    "num_clients": [4],
    "local_epochs": [1],
    "client_lr": [5e-5],
    "seed": [7],
}


def expand_sweep(base_config: ExperimentConfig, sweep: Dict[str, Sequence]):
    keys = list(sweep.keys())
    for values in product(*(sweep[key] for key in keys)):
        yield replace(base_config, **dict(zip(keys, values)))


def experiment_key(config: ExperimentConfig) -> str:
    payload = json.dumps(asdict(config), sort_keys=True, separators=(",", ":"))
    return sha256(payload.encode("utf-8")).hexdigest()[:16]


def artifact_dir_for(config: ExperimentConfig) -> Path:
    return Path(config.artifact_root) / experiment_key(config)

In [ ]:
# Load credentials from a local .env file (see .env.example).
# The notebooks read os.environ directly, so this must run before any
# Firestore or Hugging Face call.
try:
    from dotenv import load_dotenv, find_dotenv
except ImportError:
    %pip install -q python-dotenv
    from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

# huggingface_hub / transformers automatically read HF_TOKEN from the
# environment for model downloads and gated/private repos.
print("Credentials loaded:", {
    "FIREBASE_SERVICE_ACCOUNT_JSON": bool(os.environ.get("FIREBASE_SERVICE_ACCOUNT_JSON")),
    "GOOGLE_APPLICATION_CREDENTIALS": bool(os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")),
    "FIREBASE_PROJECT_ID": bool(os.environ.get("FIREBASE_PROJECT_ID")),
    "HF_TOKEN": bool(os.environ.get("HF_TOKEN")),
})

## Firestore Cache Check

The cache is checked before any fine-tuning. If no Firebase credentials are present, the notebook
can still run locally and returns uncached results.

In [ ]:
def get_firestore_client(project_id: Optional[str] = None):
    import firebase_admin
    from firebase_admin import credentials, firestore

    if not firebase_admin._apps:
        raw_json = os.environ.get("FIREBASE_SERVICE_ACCOUNT_JSON")
        cred_path = os.environ.get("GOOGLE_APPLICATION_CREDENTIALS")
        if raw_json:
            cred = credentials.Certificate(json.loads(raw_json))
        elif cred_path:
            cred = credentials.Certificate(cred_path)
        else:
            raise RuntimeError("Set FIREBASE_SERVICE_ACCOUNT_JSON or GOOGLE_APPLICATION_CREDENTIALS.")

        options = {"projectId": project_id} if project_id else None
        firebase_admin.initialize_app(cred, options=options)

    return firestore.client()


def load_cached_result(config: ExperimentConfig):
    try:
        db = get_firestore_client(os.environ.get("FIREBASE_PROJECT_ID"))
    except Exception as exc:
        return None
    snapshot = db.collection(config.firestore_collection).document(experiment_key(config)).get()
    if snapshot.exists:
        payload = snapshot.to_dict()
        if payload.get("status") == "complete":
            return payload
    return None

## Federated Fine-Tuning and the Self-Prompt Reference

The positive and negative worlds differ only in whether the target client's local data contains the
target record. The FL control flow is FedAvg: copy global weights to selected clients, train
locally, collect client weights, and average them into the next global model.

After FL fine-tuning produces the target model `theta`, this notebook builds the **self-prompt
reference `theta_dot`** *within the toy path*: it prompts the FL-fine-tuned model with short
public-domain chunks (`self_prompt_tokens` long) to "generate" a self-dataset `D_self ~ p_theta`,
then fits a second `ToyFederatedLM` on `D_self`. Because `D_self` reflects the target's generic
distribution but never contains the exact private record, the target record is a *local maximum*
under `theta` (if it was a member) but not under `theta_dot` — which is exactly what the
self-calibration exploits.

`ToyFederatedLM` gains three SPV-specific methods: `prob(text)` (a sequence likelihood in (0, 1],
`exp(-mean per-token NLL)`), `generate(prompt, ...)` (self-prompting), and `paraphrase(text, ...)`
(pure-Python symmetric paraphrases via token masking, the semantic-domain variant).

In the member world the target client sees the private record `MEMORIZATION_REPEATS` times:
memorization — and therefore the sharpness of the local maximum SPV-MIA detects — grows with
repeated exposure, which is the effect the toy model stands in for. `generate` breaks
frequency ties lexicographically so the self-dataset (and hence `theta_dot`) is deterministic and
independent of Python's string-hash randomization.

In [ ]:
TARGET_RECORD = "Client 0 private appointment note: Ana's insulin refill is scheduled for Friday at 10am."
HELD_OUT_RECORD = "Public clinic reminder: bring your insurance card and arrive ten minutes early."
CLIENT_CORPUS = [
    ["Client 0 billing question about invoice dates.", "Client 0 support chat about portal login."],
    ["Client 1 shipping update for a replacement device.", "Client 1 warranty call summary."],
    ["Client 2 product feedback about keyboard layout.", "Client 2 short troubleshooting note."],
    ["Client 3 scheduling request for a generic follow-up.", "Client 3 public FAQ paraphrase."],
]
# Short public-domain chunks used to self-prompt the target model into D_self.
PUBLIC_PROMPTS = [
    "The quarterly report summarizes operational updates for every regional office.",
    "Customers may contact the support desk regarding invoices, shipping, and returns.",
    "This document describes general scheduling and follow-up procedures for clients.",
    "Standard reminders cover appointments, billing questions, and account access.",
]

MASK_TOKEN = "<mask>"
# Repeated exposure of the private record in the member world. Memorization (a sharp
# local maximum) strengthens with exposure, which is exactly the SPV-MIA signal; the
# non-member world instead sees the held-out record once.
MEMORIZATION_REPEATS = 6


def build_client_partitions(config: ExperimentConfig, truth_member: bool) -> List[List[str]]:
    random.seed(config.seed)
    partitions = [list(records) for records in CLIENT_CORPUS[: config.num_clients]]
    while len(partitions) < config.num_clients:
        partitions.append([f"Synthetic client {len(partitions)} ordinary support record."])
    if truth_member:
        partitions[config.target_client_id].extend([TARGET_RECORD] * MEMORIZATION_REPEATS)
    else:
        partitions[config.target_client_id].append(HELD_OUT_RECORD)
    return partitions


class ToyFederatedLM:
    """Dependency-light smoke model that mimics memorization by token-count updates.

    Extends the shared toy scorer with the three SPV-MIA primitives: prob (sequence
    likelihood), generate (self-prompting), and paraphrase (symmetric masked variants).
    """

    def __init__(self):
        self.token_counts = {}

    def copy(self):
        clone = ToyFederatedLM()
        clone.token_counts = dict(self.token_counts)
        return clone

    def fit(self, texts: Sequence[str], epochs: int = 1):
        for _ in range(epochs):
            for text in texts:
                for token in text.lower().split():
                    self.token_counts[token] = self.token_counts.get(token, 0.0) + 1.0
        return self

    def nll(self, text: str) -> float:
        tokens = text.lower().split()
        if not tokens:
            return 0.0
        total = sum(self.token_counts.values()) + 1.0
        vocab = len(self.token_counts) + 1.0
        score = 0.0
        for token in tokens:
            prob = (self.token_counts.get(token, 0.0) + 1.0) / (total + vocab)
            score += -math.log(prob)
        return score / len(tokens)

    def prob(self, text: str) -> float:
        """Sequence likelihood proxy p_theta(x) = exp(-mean per-token NLL) in (0, 1].

        A memorized record (tokens with high counts) gets low NLL -> high prob; masking
        tokens to an unseen placeholder raises NLL -> lowers prob, so a memorized record
        is a LOCAL MAXIMUM relative to its paraphrases.
        """
        return math.exp(-self.nll(text))

    def generate(self, prompt: str, num_tokens: int = 8) -> str:
        """Self-prompt generation: extend a public prompt with the model's most frequent
        tokens (a toy stand-in for autoregressive sampling from p_theta).

        Ties are broken lexicographically so the self-dataset is fully deterministic and
        independent of dict/set iteration order (Python string-hash randomization)."""
        ranked = sorted(self.token_counts.items(), key=lambda kv: (-kv[1], kv[0]))
        top = [tok for tok, _ in ranked[:num_tokens]]
        return prompt if not top else prompt + " " + " ".join(top)

    def paraphrase(self, text: str, num_paraphrases: int = 4, mask_ratio: float = 0.2,
                   seed: int = 0) -> List[str]:
        """Symmetric paraphrases (semantic domain): mask ~mask_ratio of tokens with an
        unseen placeholder. Pure-Python, deterministic given the seed."""
        rng = random.Random(seed)
        tokens = text.split()
        if not tokens:
            return [text for _ in range(num_paraphrases)]
        n_mask = max(1, int(len(tokens) * mask_ratio))
        variants = []
        for _ in range(num_paraphrases):
            toks = list(tokens)
            for idx in rng.sample(range(len(toks)), min(n_mask, len(toks))):
                toks[idx] = MASK_TOKEN
            variants.append(" ".join(toks))
        return variants


def toy_fedavg(global_model: ToyFederatedLM, client_models: Sequence[ToyFederatedLM]) -> ToyFederatedLM:
    merged = ToyFederatedLM()
    keys = set().union(*(model.token_counts.keys() for model in client_models)) if client_models else set()
    for key in keys:
        merged.token_counts[key] = sum(model.token_counts.get(key, 0.0) for model in client_models) / len(client_models)
    return merged


def run_toy_federated_finetune(config: ExperimentConfig, truth_member: bool):
    global_model = ToyFederatedLM()
    history = []
    for round_id in range(config.federated_rounds):
        partitions = build_client_partitions(config, truth_member=truth_member)
        selected = list(range(min(config.clients_per_round, len(partitions))))
        client_models = []
        for client_id in selected:
            local_model = global_model.copy().fit(partitions[client_id], epochs=config.local_epochs)
            client_models.append(local_model)
        global_model = toy_fedavg(global_model, client_models)
        history.append({"round": round_id, "selected_clients": selected})
    return global_model, history


def build_self_prompt_reference_toy(config: ExperimentConfig, target_model: ToyFederatedLM):
    """Module 1 in the toy path: prompt the FL-fine-tuned target model to generate a
    self-dataset D_self, then fit a second ToyFederatedLM (theta_dot) on it."""
    self_dataset = []
    for prompt in PUBLIC_PROMPTS:
        chunk = " ".join(prompt.split()[: config.self_prompt_tokens])
        self_dataset.append(target_model.generate(chunk, num_tokens=config.self_prompt_tokens))
    reference_model = ToyFederatedLM().fit(self_dataset, epochs=1)
    return reference_model, self_dataset

### Hugging Face FL Hook

Set `use_hf_models=True` after installing `torch`, `transformers`, and `"flwr[simulation]"`. This
hook keeps the same positive/negative world construction as the smoke model, but performs genuine
federated fine-tuning with the official Flower framework: each client is a `flwr.client.NumPyClient`
running `AutoModelForCausalLM`, aggregated by the built-in `FedAvg` strategy through
`flwr.simulation.run_simulation`. The dependency-light `ToyFederatedLM` smoke path above is
unchanged. The self-prompt reference `theta_dot` for the HF path is built separately in the attack
cell (generate `D_self` from the fine-tuned model, then LoRA-fine-tune a fresh base model on it).

In [ ]:
def run_hf_federated_finetune(config: ExperimentConfig, truth_member: bool):
    """Genuine federated fine-tuning of an open-source causal LM with Flower (flwr).

    Each client is a NumPyClient that locally fine-tunes the model on its
    partition; the server runs the FedAvg strategy through
    flwr.simulation.run_simulation. Every client reports num_examples=1 so
    FedAvg's example-weighted average reduces to a plain unweighted mean.
    """
    from collections import OrderedDict

    import torch
    from torch.utils.data import DataLoader, TensorDataset
    from transformers import AutoModelForCausalLM, AutoTokenizer

    import flwr
    from flwr.client import NumPyClient, ClientApp
    from flwr.common import Context, ndarrays_to_parameters, parameters_to_ndarrays
    from flwr.server import ServerApp, ServerAppComponents, ServerConfig
    from flwr.server.strategy import FedAvg
    from flwr.simulation import run_simulation

    use_cuda = config.sim_num_gpus > 0 and torch.cuda.is_available()
    client_dev = "cuda" if use_cuda else "cpu"
    eval_dev = "cuda" if torch.cuda.is_available() else "cpu"

    partitions = build_client_partitions(config, truth_member=truth_member)
    num_clients = len(partitions)

    def get_parameters(model):
        return [value.detach().cpu().numpy() for value in model.state_dict().values()]

    def set_parameters(model, parameters):
        state_dict = OrderedDict(
            (key, torch.tensor(value)) for key, value in zip(model.state_dict().keys(), parameters)
        )
        model.load_state_dict(state_dict, strict=True)

    def load_model_and_tokenizer():
        tokenizer = AutoTokenizer.from_pretrained(config.model_id)
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
        model = AutoModelForCausalLM.from_pretrained(config.model_id)
        return model, tokenizer

    class SpvFlowerClient(NumPyClient):
        def __init__(self, partition_id, texts):
            self.partition_id = partition_id
            self.texts = texts

        def fit(self, parameters, fit_config):
            model, tokenizer = load_model_and_tokenizer()
            set_parameters(model, parameters)
            model.to(client_dev)
            model.train()
            encoded = tokenizer(
                self.texts,
                padding=True,
                truncation=True,
                max_length=config.max_length,
                return_tensors="pt",
            )
            dataset = TensorDataset(encoded["input_ids"], encoded["attention_mask"])
            loader = DataLoader(dataset, batch_size=config.local_batch_size, shuffle=True)
            optimizer = torch.optim.AdamW(model.parameters(), lr=config.client_lr)
            for _ in range(config.local_epochs):
                for input_ids, attention_mask in loader:
                    input_ids = input_ids.to(client_dev)
                    attention_mask = attention_mask.to(client_dev)
                    outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
                    outputs.loss.backward()
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
            updated = get_parameters(model)
            del model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            # num_examples=1 -> FedAvg weighted mean reduces to an unweighted mean.
            return updated, 1, {"partition_id": self.partition_id}

    init_model, _ = load_model_and_tokenizer()
    initial_parameters = ndarrays_to_parameters(get_parameters(init_model))
    del init_model

    clients_per_round = min(config.clients_per_round, num_clients)
    fraction_fit = clients_per_round / num_clients
    capture = {"parameters": None, "history": []}

    class SaveModelFedAvg(FedAvg):
        def aggregate_fit(self, server_round, results, failures):
            if results:
                selected = [int(fitres.metrics.get("partition_id", -1)) for _, fitres in results]
                capture["history"].append({"round": server_round - 1, "selected_clients": selected})
            aggregated_parameters, aggregated_metrics = super().aggregate_fit(server_round, results, failures)
            if aggregated_parameters is not None:
                capture["parameters"] = parameters_to_ndarrays(aggregated_parameters)
            return aggregated_parameters, aggregated_metrics

    def client_fn(context: Context):
        partition_id = int(context.node_config["partition-id"])
        return SpvFlowerClient(partition_id, partitions[partition_id]).to_client()

    def server_fn(context: Context):
        strategy = SaveModelFedAvg(
            fraction_fit=fraction_fit,
            fraction_evaluate=0.0,
            min_fit_clients=clients_per_round,
            min_available_clients=num_clients,
            initial_parameters=initial_parameters,
        )
        return ServerAppComponents(strategy=strategy, config=ServerConfig(num_rounds=config.federated_rounds))

    backend_config = {"client_resources": {"num_cpus": 1, "num_gpus": float(config.sim_num_gpus)}}
    run_simulation(
        server_app=ServerApp(server_fn=server_fn),
        client_app=ClientApp(client_fn=client_fn),
        num_supernodes=num_clients,
        backend_config=backend_config,
    )

    global_model, tokenizer = load_model_and_tokenizer()
    if capture["parameters"] is not None:
        set_parameters(global_model, capture["parameters"])
    global_model.to(eval_dev).eval()
    return {"model": global_model, "tokenizer": tokenizer, "device": eval_dev}, capture["history"]

## Attack Construction

The attacker computes the SPV score of the target record under the final FL model `theta` and the
self-prompt reference `theta_dot`. For each model it measures the probabilistic-variation
memorization signal `pv = prob(x) - mean(prob(paraphrases))` (sign-flipped Eq. 10 so higher =>
member), then combines them into `spv_membership_score = pv_theta - pv_theta_dot`. A memorized
record is a local maximum under `theta` (its own probability far exceeds its paraphrases') but not
under `theta_dot`, so the calibrated score is large; a non-member is flat under both, so the score
is near zero.

In [ ]:
def probabilistic_variation(prob_x: float, paraphrase_probs: Sequence[float]) -> float:
    """Probabilistic-variation memorization signal for one model.

    Paper Eq. 10 defines p_tilde(x) = mean(paraphrase_probs) - prob_x (negative for a local
    maximum / memorized record). We return the sign-flipped signal prob_x -
    mean(paraphrase_probs) = -p_tilde(x) so that HIGHER => member, matching the project
    convention. Orientation only; ranking/AUC unchanged.
    """
    if not paraphrase_probs:
        return 0.0
    return float(prob_x) - float(mean(paraphrase_probs))


def spv_membership_score(pv_theta: float, pv_theta_dot: float) -> float:
    """Self-calibrated SPV score (paper Eq. 5, flipped orientation). Higher => member."""
    return float(pv_theta) - float(pv_theta_dot)


def score_candidate_toy(target_model: ToyFederatedLM, reference_model: ToyFederatedLM,
                        text: str, config: ExperimentConfig) -> float:
    # Same paraphrase set scored under both models (paraphrasing is model-independent).
    paraphrases = target_model.paraphrase(
        text, num_paraphrases=config.num_paraphrases, mask_ratio=config.mask_ratio, seed=config.seed
    )
    pv_theta = probabilistic_variation(
        target_model.prob(text), [target_model.prob(p) for p in paraphrases]
    )
    pv_theta_dot = probabilistic_variation(
        reference_model.prob(text), [reference_model.prob(p) for p in paraphrases]
    )
    return spv_membership_score(pv_theta, pv_theta_dot)


def _mean_nll_hf(bundle, text, max_length=64):
    import torch

    model, tokenizer, device = bundle["model"], bundle["tokenizer"], bundle["device"]
    encoded = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    encoded = {key: value.to(device) for key, value in encoded.items()}
    with torch.no_grad():
        outputs = model(**encoded, labels=encoded["input_ids"])
    return float(outputs.loss.detach().cpu())


def _prob_hf(bundle, text, max_length=64):
    # Probability proxy exp(-mean per-token NLL) in (0, 1], consistent with the toy prob().
    return math.exp(-_mean_nll_hf(bundle, text, max_length=max_length))


def hf_paraphrase(text, num_paraphrases=4, mask_ratio=0.2, seed=0):
    """Pure-Python fallback paraphraser for the HF path (mask ~mask_ratio of tokens).

    Swap this for a real T5 mask-and-reconstruct paraphraser (paper default) when running
    at scale; the SPV score only needs a consistent set of symmetric paraphrases.
    """
    rng = random.Random(seed)
    tokens = text.split()
    if not tokens:
        return [text for _ in range(num_paraphrases)]
    n_mask = max(1, int(len(tokens) * mask_ratio))
    variants = []
    for _ in range(num_paraphrases):
        toks = list(tokens)
        for idx in rng.sample(range(len(toks)), min(n_mask, len(toks))):
            toks[idx] = "<mask>"
        variants.append(" ".join(toks))
    return variants


def build_self_prompt_reference_hf(config: ExperimentConfig, target_bundle):
    """Module 1 for the HF path: self-prompt the fine-tuned target to build D_self, then
    fine-tune a fresh base model on it to obtain theta_dot."""
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from torch.utils.data import DataLoader, TensorDataset

    model, tokenizer, device = target_bundle["model"], target_bundle["tokenizer"], target_bundle["device"]
    self_dataset = []
    for prompt in PUBLIC_PROMPTS:
        ids = tokenizer(prompt, return_tensors="pt", truncation=True,
                        max_length=config.self_prompt_tokens)["input_ids"].to(device)
        out = model.generate(ids, max_new_tokens=config.max_length, do_sample=True,
                             top_k=50, pad_token_id=tokenizer.eos_token_id)
        self_dataset.append(tokenizer.decode(out[0], skip_special_tokens=True))

    ref_tokenizer = AutoTokenizer.from_pretrained(config.model_id)
    if ref_tokenizer.pad_token is None:
        ref_tokenizer.pad_token = ref_tokenizer.eos_token
    ref_model = AutoModelForCausalLM.from_pretrained(config.model_id).to(device)
    encoded = ref_tokenizer(self_dataset, padding=True, truncation=True,
                            max_length=config.max_length, return_tensors="pt")
    dataset = TensorDataset(encoded["input_ids"], encoded["attention_mask"])
    loader = DataLoader(dataset, batch_size=config.local_batch_size, shuffle=True)
    optimizer = torch.optim.AdamW(ref_model.parameters(), lr=config.client_lr)
    ref_model.train()
    for _ in range(config.local_epochs):
        for input_ids, attention_mask in loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            outputs = ref_model(input_ids=input_ids, attention_mask=attention_mask, labels=input_ids)
            outputs.loss.backward()
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
    ref_model.eval()
    return {"model": ref_model, "tokenizer": ref_tokenizer, "device": device}


def score_candidate_hf(target_bundle, reference_bundle, text: str, config: ExperimentConfig) -> float:
    paraphrases = hf_paraphrase(
        text, num_paraphrases=config.num_paraphrases, mask_ratio=config.mask_ratio, seed=config.seed
    )
    pv_theta = probabilistic_variation(
        _prob_hf(target_bundle, text, config.max_length),
        [_prob_hf(target_bundle, p, config.max_length) for p in paraphrases],
    )
    pv_theta_dot = probabilistic_variation(
        _prob_hf(reference_bundle, text, config.max_length),
        [_prob_hf(reference_bundle, p, config.max_length) for p in paraphrases],
    )
    return spv_membership_score(pv_theta, pv_theta_dot)

## Attack Execution

Each trial samples a positive or negative world, runs FL fine-tuning to obtain the target model,
**builds the self-prompt reference from that target model**, then scores the target record with the
SPV signal. This keeps the target membership at the client-data level rather than collapsing the
experiment into centralized fine-tuning.

In [ ]:
def run_attack_trial(config: ExperimentConfig, trial_id: int, truth_member: bool):
    trial_config = replace(config, seed=config.seed + trial_id)
    if trial_config.use_hf_models:
        target_bundle, history = run_hf_federated_finetune(trial_config, truth_member=truth_member)
        reference_bundle = build_self_prompt_reference_hf(trial_config, target_bundle)
        score = score_candidate_hf(target_bundle, reference_bundle, TARGET_RECORD, trial_config)
    else:
        target_model, history = run_toy_federated_finetune(trial_config, truth_member=truth_member)
        reference_model, _ = build_self_prompt_reference_toy(trial_config, target_model)
        score = score_candidate_toy(target_model, reference_model, TARGET_RECORD, trial_config)

    pred_member = score >= trial_config.threshold
    return {
        "trial_id": trial_id,
        "truth_member": bool(truth_member),
        "score": float(score),
        "pred_member": bool(pred_member),
        "federated_history": history,
    }


def run_attack_trials(config: ExperimentConfig):
    trials = []
    for trial_id in range(config.attack_trials):
        truth_member = (trial_id % 2 == 0)
        trials.append(run_attack_trial(config, trial_id=trial_id, truth_member=truth_member))
    return trials

## Measurement

The primary metric follows the FL AMI project convention: `Adv = 0.5 * TPR + 0.5 * TNR`. A
threshold-free `roc_auc` is included because it is SPV-MIA's headline metric and because the SPV
score's absolute scale is model-dependent, so a fixed threshold is brittle. Other secondary
diagnostics are included for auditability.

In [ ]:
def roc_auc(labels: Sequence[bool], scores: Sequence[float]) -> float:
    pos = [s for y, s in zip(labels, scores) if y]
    neg = [s for y, s in zip(labels, scores) if not y]
    if not pos or not neg:
        return float("nan")
    wins = 0.0
    for p in pos:
        for n in neg:
            wins += 1.0 if p > n else (0.5 if p == n else 0.0)
    return wins / (len(pos) * len(neg))


def summarize_trials(trials: Sequence[Dict]):
    tp = sum(1 for row in trials if row["truth_member"] and row["pred_member"])
    tn = sum(1 for row in trials if not row["truth_member"] and not row["pred_member"])
    fp = sum(1 for row in trials if not row["truth_member"] and row["pred_member"])
    fn = sum(1 for row in trials if row["truth_member"] and not row["pred_member"])
    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    tnr = tn / (tn + fp) if (tn + fp) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tpr
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    labels = [row["truth_member"] for row in trials]
    scores = [row["score"] for row in trials]
    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tpr": tpr,
        "tnr": tnr,
        "adv": 0.5 * tpr + 0.5 * tnr,
        "accuracy": (tp + tn) / len(trials) if trials else 0.0,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc_auc(labels, scores),
        "num_trials": len(trials),
    }

## Firestore Write and Cleanup

Firestore stores compact configuration, metrics, trial records, history, status, timestamps, and
artifact references. Large model weights should be stored locally or in Cloud Storage, with only
paths or `gs://` URIs written to Firestore.

Two Firestore-specific guards are applied: per-trial round histories (lists) are wrapped inside
maps so the document contains **no directly nested arrays** (Firestore rejects those), and
`save_result` only swallows the *missing-credentials* case — a genuine serialization/write error is
re-raised so a broken document shape fails fast on the smoke run. The self-prompt reference is
recorded as `reference_model="self_prompt_reference"`.

In [ ]:
def save_result(config: ExperimentConfig, result: Dict):
    try:
        db = get_firestore_client(os.environ.get("FIREBASE_PROJECT_ID"))
    except Exception:
        # Missing-credentials case only: it is fine to skip writing locally.
        return False
    # A real write/serialization error (e.g. nested-array rejection) propagates
    # so it fails fast rather than masquerading as "not saved".
    db.collection(config.firestore_collection).document(experiment_key(config)).set(result, merge=True)
    return True


def cleanup_artifacts(artifact_dir: Path):
    artifact_dir = Path(artifact_dir)
    if artifact_dir.exists():
        shutil.rmtree(artifact_dir)


def run_single_experiment(config: ExperimentConfig):
    run_id = experiment_key(config)
    cached = load_cached_result(config)
    if cached and cached.get("status") == "complete":
        return cached

    artifact_dir = artifact_dir_for(config)
    artifact_dir.mkdir(parents=True, exist_ok=True)
    trials = run_attack_trials(config)
    metrics = summarize_trials(trials)
    result = {
        "run_id": run_id,
        "status": "complete",
        "updated_at_unix": int(time.time()),
        "config": asdict(config),
        "methodology": {
            "paper_attack": "SPV-MIA: (1) self-prompt reference model theta_dot fine-tuned on text the target LLM itself generates from short public prompts (practical difficulty calibration); (2) probabilistic variation assessment detecting whether a record is a local maximum of the model's probability via symmetric paraphrases. Decision A = 1[p_tilde_theta(x) - p_tilde_theta_dot(x) >= tau].",
            "llm_adaptation": "Positive and negative FL worlds differ by target-client membership; after Flower (flwr) FedAvg simulation produces theta, the self-prompt reference theta_dot is bootstrapped by prompting theta to generate a self-dataset and fitting a second model on it. The target record's probabilistic variation (prob(x) - mean(prob(paraphrases))) is measured under theta and theta_dot and combined as pv_theta - pv_theta_dot.",
            "metric_definition": "Adv = 0.5 * TPR + 0.5 * TNR; primary paper metric AUC (roc_auc). spv_membership_score = pv_theta - pv_theta_dot with pv = prob_x - mean(paraphrase_probs) = -p_tilde (paper Eq. 10 sign flipped so higher => member).",
            "deviation_from_source": "SPV-MIA already targets the fine-tuning phase; this transfers it to FEDERATED fine-tuning. The smoke run uses a deterministic toy causal scorer whose prob/generate/paraphrase mimic memorization and self-prompting; set use_hf_models=True for genuine federated fine-tuning of an open-source LLM (Flower FedAvg), a self-prompt reference fine-tuned on the target's generations, and (optionally) a T5 mask-and-reconstruct paraphraser in place of the pure-Python token masking.",
        },
        # Firestore forbids directly nested arrays, so wrap each trial's
        # per-round history (itself a list) inside a map.
        "federated_history": [
            {"trial_id": row["trial_id"], "rounds": row["federated_history"]}
            for row in trials
        ],
        "metrics": metrics,
        "attack_trials": [
            {key: row[key] for key in ("trial_id", "truth_member", "score", "pred_member")}
            for row in trials
        ],
        "artifacts": {
            "artifact_dir": str(artifact_dir),
            "federated_model_path": None,
            "reference_model": "self_prompt_reference",  # theta_dot bootstrapped from the target's generations.
        },
    }
    saved = save_result(config, result)
    result["firestore_saved"] = saved
    if saved and not config.keep_artifacts:
        cleanup_artifacts(artifact_dir)
    return result


def run_sweep(base_config: ExperimentConfig, sweep: Dict[str, Sequence]):
    return [run_single_experiment(config) for config in expand_sweep(base_config, sweep)]

## Smoke Run

This smoke run verifies the full ordering: Firestore cache check, FL fine-tuning, self-prompt
reference construction, adapted SPV-MIA attack execution, measurement, optional Firestore write, and
artifact cleanup. It does not download a model. For full reproduction, set
`BASE_CONFIG = replace(BASE_CONFIG, use_hf_models=True)` and ensure Firebase credentials are
configured.

The smoke run also validates the Firestore document shape (the `federated_history` is a list of
maps, not nested arrays) before any long fine-tuning job, so a persistence error cannot waste a real
run.

In [ ]:
smoke_config = replace(BASE_CONFIG, attack_trials=4, use_hf_models=False)
smoke_result = run_single_experiment(smoke_config)

assert smoke_result["metrics"]["num_trials"] == 4, smoke_result
assert any(row["truth_member"] for row in smoke_result["attack_trials"]), smoke_result
assert any(not row["truth_member"] for row in smoke_result["attack_trials"]), smoke_result
for key in ("tpr", "tnr", "adv", "roc_auc"):
    assert key in smoke_result["metrics"], smoke_result

# SPV ranking sanity: members (local maxima under theta, flat under the self-prompt
# reference theta_dot) must outrank non-members regardless of threshold.
assert smoke_result["metrics"]["roc_auc"] == 1.0, smoke_result

# Firestore shape guard: federated_history is a list of maps (no nested arrays).
fh = smoke_result["federated_history"]
assert isinstance(fh, list) and all(isinstance(item, dict) for item in fh), fh

smoke_result